# Tabela de harmonização LULC 

Harmonização das classes de uso e ocupação do solo (COS) para os anos de 1995, 2007, 2010, 2015 e 2018.

O objectivo é construir uma legenda harmonizada comum entre os vários anos, de forma a permitir a comparação temporal e a  utilização destas classes no modelo de suscetibilidade. 

* A harmonização é feita a partir do nível 4 da COS, após exclusão prévia das classes artificiais e das classes de água com base no nível 1.

In [1]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

In [2]:
fld = "/code/data/raw/lulc"
out = "/code/data/processed/centro/lulc/harmonized_tables" 
Path(out).mkdir(parents=True, exist_ok=True) 

cos = {
    1995: {
        "file": f"{fld}/COS1995v2-S1.gpkg",
        "layer": "COS1995v2",
        "n1": "COS95n1_C",
        "code": "COS95n4_C",
        "label": "COS95n4_L",
    },
    2007: {
        "file": f"{fld}/COS2007v3-S1.gpkg",
        "layer": "COS2007v3",
        "n1": "COS07n1_C",
        "code": "COS07n4_C",
        "label": "COS07n4_L",
    },
    2010: {
        "file": f"{fld}/COS2010v2-S1.gpkg",
        "layer": "COS2010v2",
        "n1": "COS10n1_C",
        "code": "COS10n4_C",
        "label": "COS10n4_L",
    },
    2015: {
        "file": f"{fld}/COS2015v2-S1.gpkg",
        "layer": "COS2015v2",
        "n1": "COS15n1_C",
        "code": "COS15n4_C",
        "label": "COS15n4_L",
    },
    2018: {
        "file": f"{fld}/COS2018v2-S1.gpkg",
        "layer": "COS2018v2",
        "n1": "COS18n1_C",
        "code": "COS18n4_C",
        "label": "COS18n4_L",
    },
}

In [3]:
#inspecionar campos
for year, cfg in cos.items():
    gdf = gpd.read_file(cfg["file"], layer=cfg["layer"], rows=5)
    print(year, gdf.columns.tolist())

1995 ['ID', 'COS95n1_C', 'COS95n1_L', 'COS95n2_C', 'COS95n2_L', 'COS95n3_C', 'COS95n3_L', 'COS95n4_C', 'COS95n4_L', 'Area_ha', 'geometry']
2007 ['ID', 'COS07n1_C', 'COS07n1_L', 'COS07n2_C', 'COS07n2_L', 'COS07n3_C', 'COS07n3_L', 'COS07n4_C', 'COS07n4_L', 'Area_ha', 'geometry']
2010 ['ID', 'COS10n1_C', 'COS10n1_L', 'COS10n2_C', 'COS10n2_L', 'COS10n3_C', 'COS10n3_L', 'COS10n4_C', 'COS10n4_L', 'Area_ha', 'geometry']
2015 ['ID', 'COS15n1_C', 'COS15n1_L', 'COS15n2_C', 'COS15n2_L', 'COS15n3_C', 'COS15n3_L', 'COS15n4_C', 'COS15n4_L', 'Area_ha', 'geometry']
2018 ['ID', 'COS18n1_C', 'COS18n1_L', 'COS18n2_C', 'COS18n2_L', 'COS18n3_C', 'COS18n3_L', 'COS18n4_C', 'COS18n4_L', 'Area_ha', 'geometry']


## Exclusão de áreas artificiais e água

Nesta fase são excluídas todas as classes cujo nível 1 (`n1`) corresponde a:

- `1` — territórios artificializados
- `9` — massas de água

In [4]:
"""classes_n4 = {}

for year in cos:
    file = cos[year]["file"]
    layer = cos[year]["layer"]
    n1 = cos[year]["n1"]
    code = cos[year]["code"]
    label = cos[year]["label"]

    gdf = gpd.read_file(file, layer=layer)[[n1, code, label]].copy()

    gdf = gdf[~gdf[n1].astype(str).isin(["1", "9"])] #níveis da cos estão em string (verificado nas Especificações Técnicas da COS)

    df = (
        gdf[[code, label]]
        .drop_duplicates()
        .sort_values([code, label])
        .reset_index(drop=True)
    )

    df.columns = ["code", "label"]
    df["year"] = year
    df = df[["year", "code", "label"]]

    classes_n4[year] = df

    print(year, len(df))
    display(df.head())"""

'classes_n4 = {}\n\nfor year in cos:\n    file = cos[year]["file"]\n    layer = cos[year]["layer"]\n    n1 = cos[year]["n1"]\n    code = cos[year]["code"]\n    label = cos[year]["label"]\n\n    gdf = gpd.read_file(file, layer=layer)[[n1, code, label]].copy()\n\n    gdf = gdf[~gdf[n1].astype(str).isin(["1", "9"])] #níveis da cos estão em string (verificado nas Especificações Técnicas da COS)\n\n    df = (\n        gdf[[code, label]]\n        .drop_duplicates()\n        .sort_values([code, label])\n        .reset_index(drop=True)\n    )\n\n    df.columns = ["code", "label"]\n    df["year"] = year\n    df = df[["year", "code", "label"]]\n\n    classes_n4[year] = df\n\n    print(year, len(df))\n    display(df.head())'

In [5]:
from pathlib import Path
import pandas as pd
import pyogrio

classes_n4 = {}

for year, cfg in cos.items():
    file = cfg["file"]
    layer = cfg["layer"]
    n1 = cfg["n1"]
    code = cfg["code"]
    label = cfg["label"]

    df = pyogrio.read_dataframe(
        file,
        layer=layer,
        columns=[n1, code, label],
        read_geometry=False,
        use_arrow=True
    )

    df = df[~df[n1].astype(str).isin(["1", "9"])]

    df = (
        df[[code, label]]
        .drop_duplicates()
        .sort_values([code, label])
        .reset_index(drop=True)
    )

    df.columns = ["code", "label"]
    df["year"] = year
    df = df[["year", "code", "label"]]

    classes_n4[year] = df

    print(year, len(df))
    display(df.head())

1995 31


,year,code,label
0,1995,2.1.1.1,Culturas temporárias de sequeiro e regadio
1,1995,2.1.1.2,Arrozais
2,1995,2.2.1.1,Vinhas
3,1995,2.2.2.1,Pomares
4,1995,2.2.3.1,Olivais


2007 38


,year,code,label
0,2007,2.1.1.1,Culturas temporárias de sequeiro e regadio
1,2007,2.1.1.2,Arrozais
2,2007,2.2.1.1,Vinhas
3,2007,2.2.2.1,Pomares
4,2007,2.2.3.1,Olivais


2010 38


,year,code,label
0,2010,2.1.1.1,Culturas temporárias de sequeiro e regadio
1,2010,2.1.1.2,Arrozais
2,2010,2.2.1.1,Vinhas
3,2010,2.2.2.1,Pomares
4,2010,2.2.3.1,Olivais


2015 38


,year,code,label
0,2015,2.1.1.1,Culturas temporárias de sequeiro e regadio
1,2015,2.1.1.2,Arrozais
2,2015,2.2.1.1,Vinhas
3,2015,2.2.2.1,Pomares
4,2015,2.2.3.1,Olivais


2018 38


,year,code,label
0,2018,2.1.1.1,Culturas temporárias de sequeiro e regadio
1,2018,2.1.1.2,Arrozais
2,2018,2.2.1.1,Vinhas
3,2018,2.2.2.1,Pomares
4,2018,2.2.3.1,Olivais


## Comparação das classes de nível 4 entre anos

In [6]:
dfs = []

for year in classes_n4:
    df = classes_n4[year][["code", "label"]].copy()
    df.columns = [f"code_{year}", "label"]
    dfs.append(df)

df_cmp = dfs[0].copy()

for df in dfs[1:]:
    df_cmp = df_cmp.merge(df, on="label", how="outer")

df_cmp = df_cmp.sort_values("label").reset_index(drop=True)

df_cmp.head(90)

,code_1995,label,code_2007,code_2010,code_2015,code_2018
0,2.3.3.1,Agricultura com espaços naturais e seminaturais,2.3.3.1,2.3.3.1,2.3.3.1,2.3.3.1
1,NaN,Agricultura protegida e viveiros,2.4.1.1,2.4.1.1,2.4.1.1,2.4.1.1
2,2.1.1.2,Arrozais,2.1.1.2,2.1.1.2,2.1.1.2,2.1.1.2
3,2.1.1.1,Culturas temporárias de sequeiro e regadio,2.1.1.1,2.1.1.1,2.1.1.1,2.1.1.1
4,NaN,Culturas temporárias e/ou pastagens melhoradas...,2.3.1.3,2.3.1.3,2.3.1.3,2.3.1.3
5,NaN,Culturas temporárias e/ou pastagens melhoradas...,2.3.1.2,2.3.1.2,2.3.1.2,2.3.1.2
6,NaN,Culturas temporárias e/ou pastagens melhoradas...,2.3.1.1,2.3.1.1,2.3.1.1,2.3.1.1
7,5.1.1.2,Florestas de azinheira,5.1.1.2,5.1.1.2,5.1.1.2,5.1.1.2
8,5.1.1.4,Florestas de castanheiro,5.1.1.4,5.1.1.4,5.1.1.4,5.1.1.4
9,NaN,Florestas de espécies invasoras,5.1.1.6,5.1.1.6,5.1.1.6,5.1.1.6


In [7]:
#Ver só as classes que aparecem em todos os anos
cols = [f"code_{year}" for year in classes_n4]

df_cmp_all = df_cmp.dropna(subset=cols).copy()
df_cmp_all = df_cmp_all.reset_index(drop=True)

df_cmp_all.head(100)

,code_1995,label,code_2007,code_2010,code_2015,code_2018
0,2.3.3.1,Agricultura com espaços naturais e seminaturais,2.3.3.1,2.3.3.1,2.3.3.1,2.3.3.1
1,2.1.1.2,Arrozais,2.1.1.2,2.1.1.2,2.1.1.2,2.1.1.2
2,2.1.1.1,Culturas temporárias de sequeiro e regadio,2.1.1.1,2.1.1.1,2.1.1.1,2.1.1.1
3,5.1.1.2,Florestas de azinheira,5.1.1.2,5.1.1.2,5.1.1.2,5.1.1.2
4,5.1.1.4,Florestas de castanheiro,5.1.1.4,5.1.1.4,5.1.1.4,5.1.1.4
5,5.1.1.5,Florestas de eucalipto,5.1.1.5,5.1.1.5,5.1.1.5,5.1.1.5
6,5.1.1.7,Florestas de outras folhosas,5.1.1.7,5.1.1.7,5.1.1.7,5.1.1.7
7,5.1.2.3,Florestas de outras resinosas,5.1.2.3,5.1.2.3,5.1.2.3,5.1.2.3
8,5.1.1.3,Florestas de outros carvalhos,5.1.1.3,5.1.1.3,5.1.1.3,5.1.1.3
9,5.1.2.1,Florestas de pinheiro bravo,5.1.2.1,5.1.2.1,5.1.2.1,5.1.2.1


In [8]:
## ver classes que faltam em pelo menos um ano
df_cmp_diff = df_cmp[df_cmp[cols].isna().any(axis=1)].copy()
df_cmp_diff = df_cmp_diff.reset_index(drop=True)

df_cmp_diff.head(100)

,code_1995,label,code_2007,code_2010,code_2015,code_2018
0,NaN,Agricultura protegida e viveiros,2.4.1.1,2.4.1.1,2.4.1.1,2.4.1.1
1,NaN,Culturas temporárias e/ou pastagens melhoradas...,2.3.1.3,2.3.1.3,2.3.1.3,2.3.1.3
2,NaN,Culturas temporárias e/ou pastagens melhoradas...,2.3.1.2,2.3.1.2,2.3.1.2,2.3.1.2
3,NaN,Culturas temporárias e/ou pastagens melhoradas...,2.3.1.1,2.3.1.1,2.3.1.1,2.3.1.1
4,NaN,Florestas de espécies invasoras,5.1.1.6,5.1.1.6,5.1.1.6,5.1.1.6
5,3.0.0.0,Pastagens,NaN,NaN,NaN,NaN
6,NaN,Pastagens espontâneas,3.1.2.1,3.1.2.1,3.1.2.1,3.1.2.1
7,NaN,Pastagens melhoradas,3.1.1.1,3.1.1.1,3.1.1.1,3.1.1.1
8,7.1.1.0,"Praias, dunas e areais",NaN,NaN,NaN,NaN
9,NaN,"Praias, dunas e areais costeiros",7.1.1.2,7.1.1.2,7.1.1.2,7.1.1.2


In [9]:
print("Total de classes na comparação:", len(df_cmp))
print("Classes presentes em todos os anos:", len(df_cmp_all))
print("Classes com diferenças entre anos:", len(df_cmp_diff))

Total de classes na comparação: 40
Classes presentes em todos os anos: 29
Classes com diferenças entre anos: 11


## Construção da tabela de harmonização

In [10]:
# juntar todas as classes
df_harm = pd.concat(classes_n4.values(), ignore_index=True)
df_harm = df_harm.sort_values(["year", "code", "label"]).reset_index(drop=True)

df_harm.columns = ["year", "code_original", "label_original"]

df_harm.head(50)

,year,code_original,label_original
0,1995,2.1.1.1,Culturas temporárias de sequeiro e regadio
1,1995,2.1.1.2,Arrozais
2,1995,2.2.1.1,Vinhas
3,1995,2.2.2.1,Pomares
4,1995,2.2.3.1,Olivais
5,1995,2.3.2.1,Mosaicos culturais e parcelares complexos
6,1995,2.3.3.1,Agricultura com espaços naturais e seminaturais
7,1995,3.0.0.0,Pastagens
8,1995,4.1.1.1,SAF de sobreiro
9,1995,4.1.1.2,SAF de azinheira


In [11]:
# criar novas colunas
df_harm["label_harmonizada"] = df_harm["label_original"]
df_harm["incluir_modelo"] = 1
df_harm["obs"] = "Mantido directamente"

## Exclusão de classes que não entram no modelo

In [12]:
# excluir classes do nivel 4 que não se enquadram no contexto
labels_excluir = [
    "Pauis",
    "Sapais",
    "Zonas entremarés",
    "Praias, dunas e areais",
    "Praias, dunas e areais costeiros",
    "Praias, dunas e areais interiores",
]

df_harm.loc[df_harm["label_original"].isin(labels_excluir), "incluir_modelo"] = 0
df_harm.loc[df_harm["label_original"].isin(labels_excluir), "obs"] = "Excluir do modelo"

## Agregação das classes 

In [13]:
# agregar pastagens
labels_pastagens = [
    "Pastagens",
    "Pastagens espontâneas",
    "Pastagens melhoradas",
]

df_harm.loc[df_harm["label_original"].isin(labels_pastagens), "label_harmonizada"] = "Pastagens"
df_harm.loc[df_harm["label_original"].isin(labels_pastagens), "obs"] = "Agregado na classe Pastagens"

In [14]:
#filtrar modelo
df_harm = df_harm[df_harm["incluir_modelo"] == 1].copy()

df_leg = (
    df_harm[["label_harmonizada"]]
    .drop_duplicates()
    .sort_values("label_harmonizada")
    .reset_index(drop=True)
)

df_leg["id_harm"] = range(1, len(df_leg) + 1)


# juntar id_harm à tabela completa
df_harm = df_harm.merge(
    df_leg[["label_harmonizada", "id_harm"]],
    on="label_harmonizada",
    how="left"
)

df_harm["id_harm"] = df_harm["id_harm"].astype("Int64")

## Exportação da tabela de harmonização

In [15]:
#guardar tabela de harmonização
df_harm.to_csv(f"{out}/tabela_harmonizacao.csv", index=False)

In [16]:
df_harm.head()

,year,code_original,label_original,label_harmonizada,incluir_modelo,obs,id_harm
0,1995,2.1.1.1,Culturas temporárias de sequeiro e regadio,Culturas temporárias de sequeiro e regadio,1,Mantido directamente,4
1,1995,2.1.1.2,Arrozais,Arrozais,1,Mantido directamente,3
2,1995,2.2.1.1,Vinhas,Vinhas,1,Mantido directamente,32
3,1995,2.2.2.1,Pomares,Pomares,1,Mantido directamente,22
4,1995,2.2.3.1,Olivais,Olivais,1,Mantido directamente,20
